# AIC 2026 B01 production ASR worker

Use exactly two GPU notebook copies. Set `WORKER_ID` to `0` in one copy and `1` in the other. The immutable plan assigns disjoint video partitions; every completed video (including no-audio videos) writes an atomic shard and receipt, so reconnects resume safely. This notebook never finalizes or imports artifacts.

In [ ]:
WORKER_ID = 0
WORKER_COUNT = 2
ASR_PROFILE = 'production'
BATCH_ID = 'aic25-b1-reference'
EXPECTED_VIDEOS = 873
EXPECTED_CANONICAL_FRAMES = 176_707
assert WORKER_ID in range(WORKER_COUNT)


In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/AIC2026/B01')
RAW_ROOT = DRIVE_ROOT / 'raw'
OUTPUT_ROOT = DRIVE_ROOT / 'out'
SOURCE_BUNDLE = DRIVE_ROOT / 'source' / 'aic2026-colab-asr-source-seekable-v2.zip'
SOURCE_SIDECAR = SOURCE_BUNDLE.with_suffix('.zip.sha256')
REPO_ROOT = Path('/content/aic2026-asr')
SCRATCH_ROOT = Path('/content/aic2026-asr-scratch')
PLAN_PATH = OUTPUT_ROOT / 'plans' / 'asr-partitions-production-v1.json'

assert SOURCE_BUNDLE.is_file() and SOURCE_SIDECAR.is_file(), 'Missing ASR source bundle or SHA-256 sidecar'
expected_source_sha = SOURCE_SIDECAR.read_text(encoding='ascii').split()[0]
source_digest = hashlib.sha256()
with SOURCE_BUNDLE.open('rb') as stream:
    for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
        source_digest.update(chunk)
assert source_digest.hexdigest() == expected_source_sha, 'ASR source fingerprint mismatch'
REPO_ROOT.mkdir(parents=True, exist_ok=True)
repo_root = REPO_ROOT.resolve()
with zipfile.ZipFile(SOURCE_BUNDLE) as archive:
    for member in archive.infolist():
        target = (REPO_ROOT / member.filename).resolve()
        if target != repo_root and repo_root not in target.parents:
            raise RuntimeError(f'Unsafe source member: {member.filename}')
        if member.is_dir():
            target.mkdir(parents=True, exist_ok=True)
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(member) as source, target.open('wb') as output:
            shutil.copyfileobj(source, output, length=1024 * 1024)
os.chdir(REPO_ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv==0.11.16'], check=True)
subprocess.run(['uv', 'python', 'install', '3.11'], check=True)
subprocess.run(['uv', 'sync', '--python', '3.11', '--group', 'competition', '--frozen'], check=True)
cuda_report = subprocess.check_output(['uv', 'run', '--frozen', 'python', '-c', 'import torch, json; print(json.dumps({"available": torch.cuda.is_available(), "name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}))'], text=True).strip()
cuda = json.loads(cuda_report)
assert cuda['available'], 'Production ASR requires a Colab GPU runtime'
assert shutil.which('ffmpeg'), 'ffmpeg is unavailable in this runtime'
subprocess.run(['uv', 'run', '--frozen', 'python', 'scripts/prepare_models.py', '--only', 'asr'], check=True)
keyframes = sorted(RAW_ROOT.glob('Keyframes_L*.zip'))
videos = sorted(RAW_ROOT.glob('Videos_L*.zip'))
assert keyframes and videos, 'B01 keyframe/video archives are missing from Drive'
if not PLAN_PATH.is_file():
    command = ['uv', 'run', '--frozen', 'python', 'scripts/build_cloud_partition_plan.py', '--maps-zip', str(RAW_ROOT / 'map-keyframes-aic25-b1.zip'), '--batch-id', BATCH_ID, '--stage', 'asr', '--worker-count', str(WORKER_COUNT), '--max-frames', '8000', '--output', str(PLAN_PATH)]
    for archive in keyframes:
        command += ['--keyframes-zip', str(archive)]
    for archive in videos:
        command += ['--videos-zip', str(archive)]
    subprocess.run(command, check=True)
print(json.dumps({'cuda': cuda, 'worker_id': WORKER_ID, 'source_sha256': expected_source_sha}, sort_keys=True))


In [ ]:
plan = json.loads(PLAN_PATH.read_text(encoding='utf-8'))
assert plan['contract'] == 'cloud-partition-plan-v1'
assert plan['batch_id'] == BATCH_ID and plan['stage'] == 'asr'
assert plan['worker_count'] == WORKER_COUNT
assert plan['expected_videos'] == EXPECTED_VIDEOS
assert plan['expected_frames'] == EXPECTED_CANONICAL_FRAMES
assigned = [part for part in plan['partitions'] if part['worker_slot'] == WORKER_ID]
assert assigned, f'No ASR partitions assigned to worker {WORKER_ID}'
first_keyframe = sorted(RAW_ROOT.glob('Keyframes_L*.zip'))[0]
print(json.dumps({'plan_fingerprint': plan['plan_fingerprint'], 'worker_id': WORKER_ID, 'partitions': len(assigned), 'worker_load': plan['worker_loads'][WORKER_ID]}, sort_keys=True), flush=True)
for part in assigned:
    command = [
        'uv', 'run', '--frozen', 'python', 'scripts/colab_provision_zip.py',
        '--keyframes-zip', str(first_keyframe),
        '--videos-zip', str(RAW_ROOT / part['archive']),
        '--maps-zip', str(RAW_ROOT / 'map-keyframes-aic25-b1.zip'),
        '--output-dir', str(OUTPUT_ROOT), '--batch-id', BATCH_ID,
        '--stages', 'asr', '--asr-profile', ASR_PROFILE,
        '--expected-videos', str(EXPECTED_VIDEOS),
        '--expected-frames', str(EXPECTED_CANONICAL_FRAMES),
        '--partition-manifest', str(PLAN_PATH),
        '--partition-id', part['partition_id'],
        '--worker-slot', str(WORKER_ID), '--worker-count', str(WORKER_COUNT),
        '--resume', '--device', 'cuda', '--ffmpeg-path', 'ffmpeg',
        '--scratch-dir', str(SCRATCH_ROOT),
    ]
    print(json.dumps({'starting': part['partition_id'], 'archive': part['archive'], 'worker_id': WORKER_ID}, sort_keys=True), flush=True)
    subprocess.run(command, check=True)
print(json.dumps({'state': 'worker_complete', 'stage': 'asr', 'worker_id': WORKER_ID, 'partitions': len(assigned), 'plan_fingerprint': plan['plan_fingerprint']}, sort_keys=True))
